In [7]:
import polars as pl
from sklearn.model_selection import train_test_split
from transformers import features_target_split

# import clean data
df = pl.read_csv("data/clean.csv")
train, test = train_test_split(df, test_size=0.3)

In [8]:
# Build Feature Engineering Pipeline
from transformers import PreprocessingPipeline, ZipCodeTransformer, AustinExperimenting, DropColumnTransformer, PCATransformer

pipeline = PreprocessingPipeline(
    [
        ZipCodeTransformer(),
        PCATransformer(),
        AustinExperimenting(),
        DropColumnTransformer()
    ]
)

In [9]:
# build model
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,

    max_depth=3,
    min_child_weight=10,

    subsample=0.8,
    colsample_bytree=0.6,

    reg_alpha=1,
    reg_lambda=1.5,

    gamma=0.1,

    random_state=42
)

In [10]:
from transformers import test_model
pl.Config.set_tbl_rows(30)
test_model(pipeline,
           model,
           train,
           test,
           "price",
           regression=True,
           show_training=True)


TRAIN
Regression Report
------------------------------
MAE  : 64856.6133
MSE  : 10480546816.0000
MRSE  : 102374.5391
R²   : 0.9213
Average Error: 0.12%

TEST
Regression Report
------------------------------
MAE  : 69333.2812
MSE  : 16236581888.0000
MRSE  : 127422.8438
R²   : 0.8813
Average Error: 0.13%
shape: (13, 2)
┌─────────────────────┬────────────┐
│ Features            ┆ Importance │
│ ---                 ┆ ---        │
│ str                 ┆ f32        │
╞═════════════════════╪════════════╡
│ grade               ┆ 0.267086   │
│ waterfront          ┆ 0.165069   │
│ PC 1                ┆ 0.142739   │
│ median_price        ┆ 0.125959   │
│ view                ┆ 0.068976   │
│ sqft_living15       ┆ 0.058394   │
│ lat                 ┆ 0.057954   │
│ long                ┆ 0.034287   │
│ yr_built            ┆ 0.030215   │
│ yr_since_renovation ┆ 0.016821   │
│ sqft_lot15          ┆ 0.01156    │
│ condition           ┆ 0.011221   │
│ sqft_lot            ┆ 0.009719   │
└─────────────

In [11]:
from transformers import regression_report

# holdout = pl.read_csv("https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/housing_holdout_test_mini.csv")
holdout = pl.read_csv("https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/housing_holdout_test.csv")
pipeline.run_train(df)
holdout_preprocessed = pipeline.run_inference(holdout)
preds = model.predict(holdout_preprocessed)
pl.DataFrame(preds, schema=["predictions"]).write_csv("data/holdout_preds.csv")